# Pricing Elasticity Model for Subscription Businesses

A practical framework for modeling demand sensitivity to price changes and finding the revenue-maximizing price point.

Covers:
- **Demand curve** — how subscriber volume responds to price
- **Price elasticity of demand** — the % drop in demand for every 1% price increase
- **Revenue curve** — which price point maximizes revenue
- **Scenario simulation** — compare revenue outcomes across price strategies
- **Paywall optimization** — conversion rate vs. price trade-off

---
*Related post: [The pricing mistake most subscription PMs make](https://www.guidomamone.com/blog-pricing-mistake.html)*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8fafc',
    'axes.grid': True,
    'grid.color': '#e2e8f0',
    'grid.linewidth': 0.8,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. The Demand Curve Model

We use a **constant-elasticity demand function**:

$$Q(P) = Q_0 \cdot \left(\frac{P}{P_0}\right)^{-\varepsilon}$$

Where:
- $Q_0$ = current subscriber count at current price $P_0$
- $\varepsilon$ (epsilon) = price elasticity of demand (always positive)
- A higher $\varepsilon$ means demand is more sensitive to price

**Elasticity interpretation:**
- $\varepsilon < 1$ — inelastic: a 10% price increase loses less than 10% of subscribers → revenue goes up
- $\varepsilon = 1$ — unit elastic: price and volume changes cancel out
- $\varepsilon > 1$ — elastic: a 10% price increase loses more than 10% of subscribers → revenue goes down

Subscription businesses with strong retention and differentiation typically have $\varepsilon$ between 0.5 and 1.5.

In [ ]:
# ── Business parameters ────────────────────────────────────
# Swap in your actual numbers here
CURRENT_PRICE      = 9.99    # current monthly price ($)
CURRENT_SUBS       = 10_000  # current subscriber count
VARIABLE_COST      = 1.50    # cost to serve one subscriber per month ($)
FIXED_COST_MONTHLY = 25_000  # monthly fixed costs ($)

# Elasticity scenarios
ELASTICITY_BASE    = 0.8     # our best estimate
ELASTICITY_LOW     = 0.5     # optimistic (less sensitive)
ELASTICITY_HIGH    = 1.3     # pessimistic (more sensitive)

# ── Demand function ────────────────────────────────────────
def demand(price, elasticity, base_price=CURRENT_PRICE, base_subs=CURRENT_SUBS):
    """Subscribers at a given price, given elasticity."""
    return base_subs * (price / base_price) ** (-elasticity)

def monthly_revenue(price, elasticity):
    return demand(price, elasticity) * price

def monthly_profit(price, elasticity):
    subs = demand(price, elasticity)
    return subs * (price - VARIABLE_COST) - FIXED_COST_MONTHLY

# Price range to evaluate
prices = np.linspace(4.99, 24.99, 300)

print(f"Current price:       ${CURRENT_PRICE}")
print(f"Current subscribers: {CURRENT_SUBS:,}")
print(f"Current MRR:         ${CURRENT_SUBS * CURRENT_PRICE:,.0f}")
print(f"Current profit:      ${monthly_profit(CURRENT_PRICE, ELASTICITY_BASE):,.0f}/mo")

## 2. Demand & Revenue Curves

These two charts answer the core question: *what happens to subscriber count and revenue if I change my price?*

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Demand curve ────────────────────────────────────────────
for e, label, color, ls in [
    (ELASTICITY_HIGH, f'ε = {ELASTICITY_HIGH} (elastic)',   '#ef4444', '--'),
    (ELASTICITY_BASE, f'ε = {ELASTICITY_BASE} (base case)', '#185FA5', '-'),
    (ELASTICITY_LOW,  f'ε = {ELASTICITY_LOW} (inelastic)',  '#22c55e', '--'),
]:
    subs = [demand(p, e) for p in prices]
    ax1.plot(prices, subs, label=label, color=color, linestyle=ls, linewidth=2)

ax1.axvline(CURRENT_PRICE, color='#94a3b8', linewidth=1, linestyle=':', label='Current price')
ax1.axhline(CURRENT_SUBS,  color='#94a3b8', linewidth=1, linestyle=':')
ax1.scatter([CURRENT_PRICE], [CURRENT_SUBS], color='#185FA5', zorder=5, s=80)
ax1.set_xlabel('Monthly price ($)', fontsize=10)
ax1.set_ylabel('Subscribers', fontsize=10)
ax1.set_title('Demand Curve', fontsize=13, fontweight='600')
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax1.legend(fontsize=9)

# ── Revenue curve ────────────────────────────────────────────
for e, label, color, ls in [
    (ELASTICITY_HIGH, f'ε = {ELASTICITY_HIGH}', '#ef4444', '--'),
    (ELASTICITY_BASE, f'ε = {ELASTICITY_BASE}', '#185FA5', '-'),
    (ELASTICITY_LOW,  f'ε = {ELASTICITY_LOW}',  '#22c55e', '--'),
]:
    rev = [monthly_revenue(p, e) for p in prices]
    ax2.plot(prices, rev, label=label, color=color, linestyle=ls, linewidth=2)

ax2.axvline(CURRENT_PRICE, color='#94a3b8', linewidth=1, linestyle=':')
current_rev = monthly_revenue(CURRENT_PRICE, ELASTICITY_BASE)
ax2.scatter([CURRENT_PRICE], [current_rev], color='#185FA5', zorder=5, s=80)
ax2.set_xlabel('Monthly price ($)', fontsize=10)
ax2.set_ylabel('Monthly revenue ($)', fontsize=10)
ax2.set_title('Revenue Curve', fontsize=13, fontweight='600')
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax2.legend(fontsize=9)

plt.suptitle('How Price Changes Affect Demand and Revenue', fontsize=14, fontweight='600', y=1.02)
plt.tight_layout()
plt.show()

## 3. Revenue-Maximizing Price

For a constant-elasticity demand curve, the revenue-maximizing price has a closed-form solution:

$$P^* = P_0 \cdot \left(\frac{\varepsilon}{\varepsilon - 1}\right) \cdot \frac{\varepsilon - 1}{\varepsilon} \quad \Rightarrow \quad P^*_{\text{revenue}} = \frac{\varepsilon}{\varepsilon - 1} \cdot c$$

But in practice, we optimize numerically — it's more flexible and handles edge cases cleanly. We find the optimal price for both **revenue** and **profit** (which accounts for variable costs).

In [ ]:
def find_optimal_price(elasticity, objective='revenue', price_range=(4.99, 49.99)):
    """Find price that maximizes revenue or profit."""
    if objective == 'revenue':
        fn = lambda p: -monthly_revenue(p, elasticity)
    else:
        fn = lambda p: -monthly_profit(p, elasticity)
    result = minimize_scalar(fn, bounds=price_range, method='bounded')
    return result.x

print("── Revenue-maximizing price ────────────────────────")
for e, label in [(ELASTICITY_LOW, 'Inelastic (ε=0.5)'),
                  (ELASTICITY_BASE, 'Base case (ε=0.8)'),
                  (ELASTICITY_HIGH, 'Elastic (ε=1.3)')]:
    opt_p  = find_optimal_price(e, 'revenue')
    opt_r  = monthly_revenue(opt_p, e)
    curr_r = monthly_revenue(CURRENT_PRICE, e)
    print(f"  {label}: ${opt_p:.2f}/mo → MRR ${opt_r:,.0f}  "
          f"(vs. current ${curr_r:,.0f}, {(opt_r/curr_r - 1):+.1%})")

print("\n── Profit-maximizing price ─────────────────────────")
for e, label in [(ELASTICITY_LOW, 'Inelastic (ε=0.5)'),
                  (ELASTICITY_BASE, 'Base case (ε=0.8)'),
                  (ELASTICITY_HIGH, 'Elastic (ε=1.3)')]:
    opt_p   = find_optimal_price(e, 'profit')
    opt_pr  = monthly_profit(opt_p, e)
    curr_pr = monthly_profit(CURRENT_PRICE, e)
    print(f"  {label}: ${opt_p:.2f}/mo → Profit ${opt_pr:,.0f}  "
          f"(vs. current ${curr_pr:,.0f}, {(opt_pr/curr_pr - 1):+.1%})")

## 4. Price Scenario Simulation

Rather than picking a single optimal price, good pricing decisions compare a small number of realistic scenarios. This table shows the full P&L impact of several price points under each elasticity assumption.

In [ ]:
price_scenarios = [7.99, 8.99, 9.99, 11.99, 13.99, 16.99, 19.99]

rows = []
for p in price_scenarios:
    for e, e_label in [(ELASTICITY_LOW, 'ε=0.5'), (ELASTICITY_BASE, 'ε=0.8'), (ELASTICITY_HIGH, 'ε=1.3')]:
        subs   = demand(p, e)
        rev    = monthly_revenue(p, e)
        profit = monthly_profit(p, e)
        rows.append({
            'Price': f'${p}',
            'Elasticity': e_label,
            'Subscribers': f'{subs:,.0f}',
            'MRR': f'${rev:,.0f}',
            'Monthly Profit': f'${profit:,.0f}',
            'MRR vs Current': f'{(rev / monthly_revenue(CURRENT_PRICE, e) - 1):+.1%}',
        })

sim_df = pd.DataFrame(rows)

# Pivot to show elasticity scenarios side by side
pivot = sim_df.pivot_table(
    index='Price', columns='Elasticity',
    values=['Subscribers', 'MRR', 'MRR vs Current'],
    aggfunc='first'
)
print(pivot.to_string())

# Visualize MRR by price and elasticity
fig, ax = plt.subplots(figsize=(12, 5))

for e, label, color, ls in [
    (ELASTICITY_HIGH, 'ε = 1.3 (elastic)',   '#ef4444', '--'),
    (ELASTICITY_BASE, 'ε = 0.8 (base)',       '#185FA5', '-'),
    (ELASTICITY_LOW,  'ε = 0.5 (inelastic)',  '#22c55e', '--'),
]:
    mrr_vals = [monthly_revenue(p, e) for p in price_scenarios]
    ax.plot(price_scenarios, mrr_vals, marker='o', label=label, color=color, linestyle=ls, linewidth=2)

ax.axvline(CURRENT_PRICE, color='#94a3b8', linestyle=':', linewidth=1.5, label=f'Current (${CURRENT_PRICE})')
ax.set_xlabel('Monthly price ($)', fontsize=10)
ax.set_ylabel('Monthly recurring revenue', fontsize=10)
ax.set_title('MRR Across Price Points and Elasticity Scenarios', fontsize=13, fontweight='600')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
ax.set_xticks(price_scenarios)
ax.set_xticklabels([f'${p}' for p in price_scenarios])
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Paywall Conversion vs. Price Trade-Off

In a freemium or paywall model, conversion rate and price have an additional interaction: lowering price doesn't just retain existing subscribers, it also converts more free users. This section models that trade-off.

We simulate a pool of free users and model how conversion rate changes with price — then find the price that maximizes new MRR from that pool.

In [ ]:
FREE_USER_POOL = 50_000  # monthly active free users exposed to paywall

def conversion_rate(price, max_cvr=0.08, price_sensitivity=0.35):
    """
    Logistic conversion curve: high CVR at low price, drops as price rises.
    max_cvr: conversion rate at price → 0
    price_sensitivity: how fast conversion drops per dollar
    """
    return max_cvr / (1 + np.exp(price_sensitivity * (price - 10)))

paywall_prices = np.linspace(2.99, 29.99, 300)
cvrs    = [conversion_rate(p) for p in paywall_prices]
new_subs = [FREE_USER_POOL * conversion_rate(p) for p in paywall_prices]
new_mrr  = [FREE_USER_POOL * conversion_rate(p) * p for p in paywall_prices]

opt_idx   = np.argmax(new_mrr)
opt_price = paywall_prices[opt_idx]
opt_mrr   = new_mrr[opt_idx]
opt_cvr   = cvrs[opt_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Conversion rate
ax1.plot(paywall_prices, [c * 100 for c in cvrs], color='#185FA5', linewidth=2.5)
ax1.axvline(opt_price, color='#f59e0b', linewidth=1.5, linestyle='--', label=f'Revenue-max (${opt_price:.2f})')
ax1.scatter([opt_price], [opt_cvr * 100], color='#f59e0b', zorder=5, s=100)
ax1.set_xlabel('Monthly price ($)', fontsize=10)
ax1.set_ylabel('Conversion rate (%)', fontsize=10)
ax1.set_title('Paywall Conversion Rate vs. Price', fontsize=13, fontweight='600')
ax1.legend(fontsize=9)

# New MRR from paywall
ax2.plot(paywall_prices, [m/1000 for m in new_mrr], color='#185FA5', linewidth=2.5)
ax2.fill_between(paywall_prices, [m/1000 for m in new_mrr], alpha=0.12, color='#185FA5')
ax2.axvline(opt_price, color='#f59e0b', linewidth=1.5, linestyle='--',
            label=f'Optimal: ${opt_price:.2f} → ${opt_mrr/1000:.1f}k MRR')
ax2.scatter([opt_price], [opt_mrr/1000], color='#f59e0b', zorder=5, s=100)
ax2.set_xlabel('Monthly price ($)', fontsize=10)
ax2.set_ylabel('New MRR from paywall ($k)', fontsize=10)
ax2.set_title('New MRR from Paywall Conversions', fontsize=13, fontweight='600')
ax2.legend(fontsize=9)

plt.suptitle(f'Paywall Optimization  |  Free pool: {FREE_USER_POOL:,} users',
             fontsize=14, fontweight='600', y=1.02)
plt.tight_layout()
plt.show()

print(f"Revenue-maximizing paywall price: ${opt_price:.2f}")
print(f"Conversion rate at that price:    {opt_cvr:.2%}")
print(f"New subscribers per month:        {FREE_USER_POOL * opt_cvr:,.0f}")
print(f"New MRR from paywall:             ${opt_mrr:,.0f}")

## 6. Key Takeaways

**What this model tells you:**

1. **You can't optimize price without estimating elasticity.** The right price under ε=0.5 (inelastic) is much higher than under ε=1.3 (elastic). Run surveys or price experiments to narrow your elasticity range before making a big price move.

2. **Revenue-maximizing ≠ profit-maximizing.** When you account for variable cost (cost to serve), the optimal price is always higher than the revenue-max price. Many teams optimize the wrong objective.

3. **Paywall pricing is different from renewal pricing.** Conversion rate at the top of the funnel introduces an additional lever — you can trade conversion volume for ARPU. The optimal paywall price is typically lower than the renewal price.

### To adapt this to your own data
1. Replace `CURRENT_PRICE`, `CURRENT_SUBS`, `VARIABLE_COST`, `FIXED_COST_MONTHLY` with your actuals
2. Calibrate `ELASTICITY_BASE` from historical price tests or conjoint surveys
3. In the paywall section, replace `conversion_rate()` with your empirical CVR data at different price points

---
*Built by [Guido Mamone](https://www.guidomamone.com) — PM specializing in subscription monetization and growth.*